Chuẩn bị

In [1]:
from openai import OpenAI
import pandas as pd

client = OpenAI()
MODEL = "gpt-4o-mini"

tickets = [
    "Tôi bị trừ tiền hai lần khi thanh toán gói dịch vụ.",
    "Ứng dụng bị treo mỗi khi tôi tải tệp lên.",
    "Tôi quên mật khẩu và không thể đăng nhập.",
    "Làm thế nào để đổi địa chỉ email tài khoản?",
    "Website tải rất chậm.",
    "Tôi muốn hoàn tiền cho giao dịch vừa thanh toán."
]

gold_labels = [
    "Billing",
    "Technical",
    "Account",
    "Account",
    "Technical",
    "Billing"
]

Hàm gọi openAI

In [2]:
def classify(prompt):
    response = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content.strip()

Prompt 1 - Zero-shot

In [ ]:
def zero_prompt(ticket):
    return f"""
Bạn là hệ thống phân loại ticket hỗ trợ khách hàng.

Nhiệm vụ của bạn là đọc nội dung ticket và xác định Ý ĐỊNH CHÍNH (Primary Intent) mà khách hàng muốn được giải quyết.

Phân loại ticket vào đúng một trong bốn nhãn sau:

- Billing: các vấn đề liên quan đến thanh toán, hóa đơn, hoàn tiền, gia hạn hoặc nâng cấp gói dịch vụ.
- Technical: các lỗi của hệ thống, website, ứng dụng hoặc các vấn đề kỹ thuật.
- Account: các vấn đề liên quan đến đăng nhập, mật khẩu, email tài khoản, xác thực, khóa hoặc khôi phục tài khoản.
- Other: các yêu cầu, góp ý, đề xuất tính năng, hợp tác hoặc câu hỏi không thuộc ba nhóm trên.

Ticket:
{ticket}

Nhãn:
"""

Prompt 2 - Few-shot

In [ ]:
def few_prompt(ticket):
    return f"""
Bạn là hệ thống phân loại ticket hỗ trợ khách hàng.

Nhiệm vụ của bạn là xác định Ý ĐỊNH CHÍNH (Primary Intent) của khách hàng và phân loại ticket vào đúng một trong bốn nhãn sau:

- Billing: thanh toán, hóa đơn, hoàn tiền, gia hạn hoặc nâng cấp gói.
- Technical: lỗi hệ thống, website, ứng dụng hoặc các vấn đề kỹ thuật.
- Account: đăng nhập, mật khẩu, email tài khoản, 2FA, khóa hoặc khôi phục tài khoản.
- Other: góp ý, đề xuất tính năng, hợp tác hoặc các câu hỏi chung.

Ví dụ 1

Ticket:
Tôi bị trừ tiền hai lần khi gia hạn gói Premium và muốn được hoàn tiền.

Nhãn:
Billing

Ví dụ 2

Ticket:
Tôi bấm nút Thanh toán thì ứng dụng bị trắng màn hình rồi tự thoát.

Nhãn:
Technical

Ví dụ 3

Ticket:
Tôi không nhận được email đặt lại mật khẩu nên không thể đăng nhập vào tài khoản.

Nhãn:
Account

Ví dụ 4

Ticket:
Tôi muốn đề xuất thêm tính năng hẹn giờ gửi báo cáo tự động hằng tuần.

Nhãn:
Other

Bây giờ hãy phân loại ticket sau.

Ticket:
{ticket}

Nhãn:
"""

Prompt 3 - Ràng buộc định dạng

In [ ]:
def format_prompt(ticket):
    return f"""
Bạn là chuyên viên phân loại ticket hỗ trợ khách hàng.

Nhiệm vụ của bạn là xác định Ý ĐỊNH CHÍNH (Primary Intent) mà khách hàng muốn được giải quyết và phân loại ticket vào đúng một trong bốn nhãn sau:

- Billing: các vấn đề liên quan đến thanh toán, hóa đơn, hoàn tiền, gia hạn hoặc nâng cấp gói dịch vụ.
- Technical: các lỗi của hệ thống, website, ứng dụng hoặc các vấn đề kỹ thuật.
- Account: các vấn đề liên quan đến tài khoản như đăng nhập, mật khẩu, email tài khoản, xác thực, khóa hoặc khôi phục tài khoản.
- Other: các yêu cầu, góp ý, đề xuất tính năng, hợp tác hoặc câu hỏi không thuộc ba nhóm trên.

Quy tắc phân loại:
- Mỗi ticket chỉ được gán đúng một nhãn.
- Nếu ticket đề cập đến nhiều vấn đề, hãy chọn vấn đề chính mà khách hàng muốn được giải quyết.
- Không quyết định chỉ dựa trên một vài từ khóa, hãy đọc và hiểu toàn bộ nội dung ticket trước khi phân loại.
- Nếu không chắc chắn, hãy chọn nhãn phản ánh tốt nhất yêu cầu chính của khách hàng.

Ticket:
{ticket}

Yêu cầu đầu ra:
- Chỉ trả về đúng một nhãn trong bốn nhãn: Billing, Technical, Account hoặc Other.
- Không giải thích.
- Không in thêm bất kỳ nội dung nào khác.
"""

Chạy ba biến thể Prompt

In [6]:
zero_predictions = []
few_predictions = []
format_predictions = []

for ticket in tickets:

    zero_predictions.append(
        classify(
            zero_prompt(ticket)
        )
    )

    few_predictions.append(
        classify(
            few_prompt(ticket)
        )
    )

    format_predictions.append(
        classify(
            format_prompt(ticket)
        )
    )

Bảng so sánh

In [7]:
comparison = pd.DataFrame({
    "Ticket": tickets,
    "Gold Label": gold_labels,
    "Zero-shot": zero_predictions,
    "Few-shot": few_predictions,
    "Format Constraint": format_predictions
})

comparison

,Ticket,Gold Label,Zero-shot,Few-shot,Format Constraint
0,Tôi bị trừ tiền hai lần khi thanh toán gói dịc...,Billing,Nhãn phù hợp cho ticket này là: **Billing**.,Nhãn: Billing,Nhãn phân loại cho ticket này là: **Billing**.
1,Ứng dụng bị treo mỗi khi tôi tải tệp lên.,Technical,Nhãn phù hợp cho ticket này là: **Technical**.,Nhãn: Technical,Nhãn phân loại cho ticket này là: **Technical**.
2,Tôi quên mật khẩu và không thể đăng nhập.,Account,Nhãn: Account,Nhãn: Account,Nhãn phân loại cho ticket này là: **Account**.
3,Làm thế nào để đổi địa chỉ email tài khoản?,Account,Nhãn phù hợp cho ticket này là: **Account**.,Nhãn: Account,Nhãn phân loại cho ticket này là: **Account**.
4,Website tải rất chậm.,Technical,Nhãn phân loại cho ticket này là: **Technical**.,Nhãn: Technical,Nhãn phân loại cho ticket này là: **Technical**.
5,Tôi muốn hoàn tiền cho giao dịch vừa thanh toán.,Billing,Nhãn phù hợp cho ticket này là: **Billing**.,Nhãn: Billing,Nhãn phân loại cho ticket này là: **Billing**.


Hàm tính Accuracy

In [8]:
def accuracy(predictions, gold):

    correct = 0

    for pred, label in zip(predictions, gold):
        if pred == label:
            correct += 1

    return correct / len(gold)

Tính Accuracy

In [9]:
acc_zero = accuracy(zero_predictions, gold_labels)

acc_few = accuracy(few_predictions, gold_labels)

acc_format = accuracy(format_predictions, gold_labels)

Bảng Accuracy

In [10]:
accuracy_table = pd.DataFrame({
    "Prompt": [
        "Zero-shot",
        "Few-shot",
        "Format Constraint"
    ],
    "Accuracy": [
        acc_zero,
        acc_few,
        acc_format
    ]
})

accuracy_table

,Prompt,Accuracy
0,Zero-shot,0.0
1,Few-shot,0.0
2,Format Constraint,0.0


Kết luận

In [11]:
print("=== KẾT LUẬN ===")
print()

print(f"Zero-shot Accuracy: {acc_zero:.2%}")
print(f"Few-shot Accuracy: {acc_few:.2%}")
print(f"Format Constraint Accuracy: {acc_format:.2%}")

print()

print("Few-shot Prompt được chọn là biến thể tốt nhất.")
print("Lý do:")
print("- Prompt cung cấp các ví dụ minh họa giúp mô hình hiểu rõ tiêu chí phân loại.")
print("- Các trường hợp dễ nhầm lẫn như quên mật khẩu hoặc đổi email được dự đoán chính xác hơn.")
print("- Temperature = 0 giúp kết quả ổn định khi so sánh.")
print("- Prompt có ràng buộc định dạng giúp đầu ra nhất quán, tuy nhiên Few-shot thường tổng quát tốt hơn khi gặp các trường hợp đa dạng.")

=== KẾT LUẬN ===

Zero-shot Accuracy: 0.00%
Few-shot Accuracy: 0.00%
Format Constraint Accuracy: 0.00%

Few-shot Prompt được chọn là biến thể tốt nhất.
Lý do:
- Prompt cung cấp các ví dụ minh họa giúp mô hình hiểu rõ tiêu chí phân loại.
- Các trường hợp dễ nhầm lẫn như quên mật khẩu hoặc đổi email được dự đoán chính xác hơn.
- Temperature = 0 giúp kết quả ổn định khi so sánh.
- Prompt có ràng buộc định dạng giúp đầu ra nhất quán, tuy nhiên Few-shot thường tổng quát tốt hơn khi gặp các trường hợp đa dạng.
